# Fine-Tuning: ResNet34 U-Net on Walmart Parking Data

Starts from the checkpoint trained on PKLot (`resnet34_unet_best.pth`) and fine-tunes on
the **walmart-1** dataset — a labelled Walmart parking-lot dataset from Roboflow.

| Stage | Dataset | Format | Classes |
|---|---|---|---|
| Pre-training (done) | PKLot | COCO JSON | background / space-empty / space-occupied |
| **Fine-tuning (this notebook)** | walmart-1 | YOLO | car / free_space |
| Testing | walmart_test | PNG (satellite) | — (unlabelled) |

## Class mapping (YOLO → model)

| YOLO label | YOLO id | Model class | Model id |
|---|---|---|---|
| `free_space` | 1 | `space-empty` | 1 |
| `car` | 0 | `space-occupied` | 2 |
| *(no bbox)* | — | `background` | 0 |

## Fine-tuning strategy

The walmart-1 dataset is small (40 train images). To avoid catastrophic forgetting:
- **Encoder frozen** during the first phase — only the decoder and segmentation head learn.
- **Full unfreezing** in a second phase with a much lower LR to let the encoder adapt gently.

In [ ]:
%pip install -q segmentation-models-pytorch albumentations timm

In [ ]:
%matplotlib inline

import re
import time
from datetime import datetime
from pathlib import Path

import albumentations as A
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import segmentation_models_pytorch as smp

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {DEVICE}")

## 1 · Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
WALMART1_ROOT  = Path("../../Data/walmart-1")
WALMART_TEST   = Path("../../Data/walmart_test")
CHECKPOINT_DIR = Path("../../checkpoints")
PRETRAINED     = CHECKPOINT_DIR / "resnet34_unet_best.pth"
FT_CHECKPOINT  = CHECKPOINT_DIR / "resnet34_unet_walmart_ft.pth"

# ── Model I/O ──────────────────────────────────────────────────────────────
IN_CHANNELS = 3
NUM_CLASSES = 3   # 0=background  1=space-empty  2=space-occupied
IMG_SIZE    = 256

# ── Training ───────────────────────────────────────────────────────────────
# Phase 1: encoder frozen, decoder only
EPOCHS_PHASE1   = 20
LR_PHASE1       = 1e-3
# Phase 2: full network, gentle unfreeze
EPOCHS_PHASE2   = 15
LR_PHASE2       = 5e-5

BATCH_SIZE      = 8
ES_PATIENCE     = 8

# ── Class mapping: YOLO id → model id ──────────────────────────────────────
# YOLO: 0=car, 1=free_space
# Model: 0=background, 1=space-empty, 2=space-occupied
YOLO_TO_MODEL = {0: 2, 1: 1}   # car→occupied, free_space→empty

# ── Visualisation ──────────────────────────────────────────────────────────
CLASS_NAMES  = ["background", "space-empty", "space-occupied"]
CLASS_COLORS = np.array([
    [0,   0,   0  ],   # black  – background
    [0,   200, 0  ],   # green  – empty
    [220, 50,  50 ],   # red    – occupied
], dtype=np.uint8)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

assert PRETRAINED.exists(), f"Pretrained checkpoint not found: {PRETRAINED}"
print(f"Pretrained checkpoint : {PRETRAINED}")
print(f"Fine-tuned checkpoint : {FT_CHECKPOINT}")

## 2 · Dataset (YOLO → pixel mask)

Each YOLO `.txt` label file contains one row per bounding box:
```
class_id  cx  cy  w  h   # all values normalised 0–1
```
We convert each box to pixel coordinates and fill that rectangle in a `(H, W)` integer mask
using the mapped model class id (1 or 2). Pixels not covered by any box remain 0 (background).

In [ ]:
class WalmartYOLODataset(Dataset):
    """
    Reads images + YOLO label files from one of the walmart-1 splits.

    split_dir must contain:
        images/   *.jpg  (or *.png)
        labels/   *.txt  (YOLO format, same stem as image)
    """

    def __init__(self, split_dir: Path, transform=None, img_size: int = 256):
        self.img_dir   = split_dir / "images"
        self.lbl_dir   = split_dir / "labels"
        self.transform = transform
        self.img_size  = img_size

        self.samples = sorted([
            p for p in self.img_dir.iterdir()
            if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        ])

    def __len__(self):
        return len(self.samples)

    def _load_mask(self, img_path: Path, orig_h: int, orig_w: int) -> np.ndarray:
        lbl_path = self.lbl_dir / (img_path.stem + ".txt")
        mask = np.zeros((orig_h, orig_w), dtype=np.uint8)
        if not lbl_path.exists():
            return mask
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                yolo_cls = int(parts[0])
                model_cls = YOLO_TO_MODEL.get(yolo_cls, 0)
                if model_cls == 0:
                    continue
                cx, cy, bw, bh = map(float, parts[1:5])
                x1 = int((cx - bw / 2) * orig_w)
                y1 = int((cy - bh / 2) * orig_h)
                x2 = int((cx + bw / 2) * orig_w)
                y2 = int((cy + bh / 2) * orig_h)
                x1, y1 = max(x1, 0), max(y1, 0)
                x2, y2 = min(x2, orig_w), min(y2, orig_h)
                mask[y1:y2, x1:x2] = model_cls
        return mask

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        image    = np.array(Image.open(img_path).convert("RGB"))
        orig_h, orig_w = image.shape[:2]

        mask = self._load_mask(img_path, orig_h, orig_w)

        # resize to model input size
        image = np.array(Image.fromarray(image).resize((self.img_size, self.img_size), Image.BILINEAR))
        mask  = np.array(Image.fromarray(mask).resize((self.img_size, self.img_size), Image.NEAREST))

        if self.transform:
            out   = self.transform(image=image, mask=mask)
            image = out["image"]
            mask  = out["mask"].long()
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
            mask  = torch.from_numpy(mask).long()

        return image, mask

## 3 · Augmentation & DataLoaders

The walmart-1 training set has only 40 images, so we use aggressive augmentation to avoid overfitting.

In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=15, p=0.4),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

train_ds = WalmartYOLODataset(WALMART1_ROOT / "train", transform=train_transform, img_size=IMG_SIZE)
val_ds   = WalmartYOLODataset(WALMART1_ROOT / "valid", transform=val_transform,   img_size=IMG_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train images : {len(train_ds)}")
print(f"Val   images : {len(val_ds)}")

In [ ]:
def mask_to_rgb(mask: np.ndarray) -> np.ndarray:
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls_id, color in enumerate(CLASS_COLORS):
        rgb[mask == cls_id] = color
    return rgb

def denormalise(tensor: torch.Tensor) -> np.ndarray:
    mean = np.array(IMAGENET_MEAN).reshape(3, 1, 1)
    std  = np.array(IMAGENET_STD).reshape(3, 1, 1)
    img  = tensor.cpu().numpy() * std + mean
    return np.clip(img.transpose(1, 2, 0), 0, 1)

# Sanity-check: visualise one batch
images, masks = next(iter(train_loader))
print(f"Image batch : {images.shape}  dtype={images.dtype}")
print(f"Mask  batch : {masks.shape}   dtype={masks.dtype}")
print(f"Classes in batch : {masks.unique().tolist()}")

n_show = min(4, len(images))
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
for i in range(n_show):
    axes[0, i].imshow(denormalise(images[i]))
    axes[0, i].set_title("Image"); axes[0, i].axis("off")
    axes[1, i].imshow(mask_to_rgb(masks[i].numpy()))
    axes[1, i].set_title("Mask");  axes[1, i].axis("off")

legend = [mpatches.Patch(color=CLASS_COLORS[i]/255, label=CLASS_NAMES[i]) for i in range(NUM_CLASSES)]
fig.legend(handles=legend, loc="lower center", ncol=3, fontsize=11)
plt.tight_layout()
plt.show()

## 4 · Load Pre-trained Model

In [ ]:
model = smp.Unet(
    encoder_name    = "resnet34",
    encoder_weights = None,        # weights come from our checkpoint
    in_channels     = IN_CHANNELS,
    classes         = NUM_CLASSES,
    activation      = None,
)
model.load_state_dict(torch.load(PRETRAINED, map_location=DEVICE))
model = model.to(DEVICE)
print(f"Loaded pretrained weights from: {PRETRAINED}")

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

## 5 · Loss & Metrics

In [ ]:
criterion = nn.CrossEntropyLoss()

def compute_miou(logits: torch.Tensor, targets: torch.Tensor, num_classes: int = NUM_CLASSES) -> float:
    preds = logits.argmax(dim=1)
    ious  = []
    for cls in range(num_classes):
        inter = ((preds == cls) & (targets == cls)).sum().float()
        union = ((preds == cls) | (targets == cls)).sum().float()
        ious.append(((inter + 1e-6) / (union + 1e-6)).item())
    return float(np.mean(ious))

def compute_fg_miou(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    ious  = []
    for cls in [1, 2]:
        inter = ((preds == cls) & (targets == cls)).sum().float()
        union = ((preds == cls) | (targets == cls)).sum().float()
        ious.append(((inter + 1e-6) / (union + 1e-6)).item())
    return float(np.mean(ious))

def run_epoch(loader, train: bool, optimizer=None):
    model.train() if train else model.eval()
    loss_sum = miou_sum = fg_sum = n = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, masks_gt in loader:
            imgs, masks_gt = imgs.to(DEVICE), masks_gt.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, masks_gt)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            loss_sum += loss.item()
            miou_sum += compute_miou(logits, masks_gt)
            fg_sum   += compute_fg_miou(logits, masks_gt)
            n        += 1
    return loss_sum / n, miou_sum / n, fg_sum / n

## 6 · Phase 1 — Decoder-Only Fine-Tuning

Freeze the ResNet-34 encoder. Only the U-Net decoder and segmentation head are updated.
This is safe with a small dataset: the encoder's ImageNet + PKLot features are preserved.

In [ ]:
# Freeze encoder
for param in model.encoder.parameters():
    param.requires_grad = False

frozen_params    = sum(p.numel() for p in model.encoder.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Encoder parameters frozen : {frozen_params:,}")
print(f"Trainable parameters      : {trainable_params:,}  (decoder + head)")

optimizer_p1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_PHASE1
)
scheduler_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_p1, mode="max", factor=0.5, patience=3
)

In [ ]:
history_p1 = {"train_loss": [], "val_loss": [], "val_miou": [], "val_fg_miou": []}
best_miou_p1 = 0.0
es_counter   = 0

print("Phase 1 — decoder only (encoder frozen)")
print("─" * 80)

for epoch in range(1, EPOCHS_PHASE1 + 1):
    t0 = time.time()
    train_loss, _, _          = run_epoch(train_loader, train=True,  optimizer=optimizer_p1)
    val_loss, val_miou, val_fg = run_epoch(val_loader,   train=False)
    scheduler_p1.step(val_miou)

    history_p1["train_loss"].append(train_loss)
    history_p1["val_loss"].append(val_loss)
    history_p1["val_miou"].append(val_miou)
    history_p1["val_fg_miou"].append(val_fg)

    tag = ""
    if val_miou > best_miou_p1:
        best_miou_p1 = val_miou
        torch.save(model.state_dict(), FT_CHECKPOINT)
        es_counter = 0
        tag = "  ← best"
    else:
        es_counter += 1
        tag = f"  (no improvement {es_counter}/{ES_PATIENCE})"

    print(
        f"Epoch {epoch:02d}/{EPOCHS_PHASE1}  "
        f"train_loss={train_loss:.4f}  "
        f"val_loss={val_loss:.4f}  "
        f"val_mIoU={val_miou:.4f}  "
        f"val_fg_mIoU={val_fg:.4f}  "
        f"[{time.time()-t0:.0f}s]{tag}"
    )
    if es_counter >= ES_PATIENCE:
        print(f"\nEarly stopping after epoch {epoch}.")
        break

print(f"\nPhase 1 best val mIoU: {best_miou_p1:.4f}")

## 7 · Phase 2 — Full Network Fine-Tuning

Unfreeze the encoder with a very low learning rate (`5e-5`). The decoder keeps its Phase 1 weights.
This lets the encoder gently adapt its features to Walmart parking-lot imagery.

In [ ]:
# Load best Phase 1 checkpoint before unfreezing
model.load_state_dict(torch.load(FT_CHECKPOINT, map_location=DEVICE))
print(f"Loaded Phase 1 best checkpoint.")

# Unfreeze everything
for param in model.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"All {trainable_params:,} parameters trainable.")

# Differential LR: lower for encoder, higher for decoder
optimizer_p2 = torch.optim.Adam([
    {"params": model.encoder.parameters(), "lr": LR_PHASE2 * 0.1},
    {"params": model.decoder.parameters(), "lr": LR_PHASE2},
    {"params": model.segmentation_head.parameters(), "lr": LR_PHASE2},
])
scheduler_p2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_p2, mode="max", factor=0.5, patience=3
)

In [ ]:
history_p2 = {"train_loss": [], "val_loss": [], "val_miou": [], "val_fg_miou": []}
best_miou_p2 = best_miou_p1
es_counter   = 0

print("Phase 2 — full network (encoder unfrozen, differential LR)")
print("─" * 80)

for epoch in range(1, EPOCHS_PHASE2 + 1):
    t0 = time.time()
    train_loss, _, _          = run_epoch(train_loader, train=True,  optimizer=optimizer_p2)
    val_loss, val_miou, val_fg = run_epoch(val_loader,   train=False)
    scheduler_p2.step(val_miou)

    history_p2["train_loss"].append(train_loss)
    history_p2["val_loss"].append(val_loss)
    history_p2["val_miou"].append(val_miou)
    history_p2["val_fg_miou"].append(val_fg)

    tag = ""
    if val_miou > best_miou_p2:
        best_miou_p2 = val_miou
        torch.save(model.state_dict(), FT_CHECKPOINT)
        es_counter = 0
        tag = "  ← best"
    else:
        es_counter += 1
        tag = f"  (no improvement {es_counter}/{ES_PATIENCE})"

    print(
        f"Epoch {epoch:02d}/{EPOCHS_PHASE2}  "
        f"train_loss={train_loss:.4f}  "
        f"val_loss={val_loss:.4f}  "
        f"val_mIoU={val_miou:.4f}  "
        f"val_fg_mIoU={val_fg:.4f}  "
        f"[{time.time()-t0:.0f}s]{tag}"
    )
    if es_counter >= ES_PATIENCE:
        print(f"\nEarly stopping after epoch {epoch}.")
        break

print(f"\nBest val mIoU overall: {best_miou_p2:.4f}")
print(f"Fine-tuned checkpoint saved to: {FT_CHECKPOINT}")

## 8 · Training Curves

In [ ]:
# Concatenate Phase 1 + Phase 2 histories
combined = {k: history_p1[k] + history_p2[k] for k in history_p1}
n_p1   = len(history_p1["val_miou"])
n_p2   = len(history_p2["val_miou"])
epochs = range(1, n_p1 + n_p2 + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, combined["train_loss"], label="Train loss")
ax1.plot(epochs, combined["val_loss"],   label="Val loss")
ax1.axvline(n_p1 + 0.5, color="gray", linestyle="--", linewidth=1, label="Phase boundary")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Loss curves"); ax1.legend(); ax1.grid(True)

ax2.plot(epochs, combined["val_miou"],    label="Val mIoU (all)")
ax2.plot(epochs, combined["val_fg_miou"], label="Val mIoU (fg)")
ax2.axvline(n_p1 + 0.5, color="gray", linestyle="--", linewidth=1, label="Phase boundary")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("mIoU")
ax2.set_title("Validation mIoU"); ax2.legend(); ax2.grid(True)

fig.text(0.25, 0.01, "◀ Phase 1: decoder only", ha="center", fontsize=10, color="gray")
fig.text(0.75, 0.01, "Phase 2: full network ▶", ha="center", fontsize=10, color="gray")

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / "fine_tuning_curves.png", dpi=120)
plt.show()

## 9 · Load Best Fine-Tuned Checkpoint

In [ ]:
model.load_state_dict(torch.load(FT_CHECKPOINT, map_location=DEVICE))
model.eval()
print(f"Loaded best fine-tuned checkpoint: {FT_CHECKPOINT}")

## 10 · Test on Walmart Satellite Images (`walmart_test`)

These are the real-world satellite images (NAIP, via Google Earth Engine) of a Walmart
parking lot taken at different years. They have **no labels** — evaluation is visual and
through proxy metrics (coverage %, occupancy rate).

In [ ]:
preprocess = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

def mask_to_rgba(mask: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    rgba = np.zeros((*mask.shape, 4), dtype=np.uint8)
    for cls_id, color in enumerate(CLASS_COLORS):
        if cls_id == 0:
            continue
        rgba[mask == cls_id, :3] = color
        rgba[mask == cls_id,  3] = int(255 * alpha)
    return rgba

def run_inference(img_path: Path):
    orig    = np.array(Image.open(img_path).convert("RGB"))
    h0, w0  = orig.shape[:2]
    resized = np.array(Image.fromarray(orig).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
    tensor  = preprocess(image=resized)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
    pred_small = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
    pred_full  = np.array(Image.fromarray(pred_small).resize((w0, h0), Image.NEAREST))
    return orig, pred_full

def proxy_metrics(mask: np.ndarray) -> dict:
    n       = mask.size
    parking = int((mask > 0).sum())
    return {
        "coverage_pct": parking / n * 100,
        "occ_rate_pct": int((mask == 2).sum()) / max(parking, 1) * 100,
        "bg_pct"      : int((mask == 0).sum()) / n * 100,
        "empty_pct"   : int((mask == 1).sum()) / n * 100,
        "occupied_pct": int((mask == 2).sum()) / n * 100,
    }

# Load and sort images by date
img_paths = sorted(WALMART_TEST.glob("*.png"))
records   = []
for p in img_paths:
    m     = re.search(r"(\d{4}-\d{2}-\d{2})", p.stem)
    date  = datetime.strptime(m.group(1), "%Y-%m-%d").date() if m else None
    label = m.group(1) if m else p.stem
    records.append({"path": p, "date": date, "label": label})

print(f"Found {len(records)} satellite images in walmart_test:")
for r in records:
    print(f"  {r['path'].name}  →  {r['date']}")

In [ ]:
results = []
for rec in records:
    orig, pred_mask = run_inference(rec["path"])
    m = proxy_metrics(pred_mask)
    results.append({**rec, "orig": orig, "pred_mask": pred_mask, **m})
    print(f"[{rec['label']}]  coverage={m['coverage_pct']:.1f}%  occ_rate={m['occ_rate_pct']:.1f}%")

## 11 · Visual Predictions on Satellite Images

In [ ]:
legend_patches = [
    mpatches.Patch(color=CLASS_COLORS[i] / 255, label=CLASS_NAMES[i])
    for i in range(NUM_CLASSES)
]

for rec in results:
    orig, pred_mask = rec["orig"], rec["pred_mask"]

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    fig.suptitle(
        f"Fine-tuned model — Walmart Parking Lot ({rec['label']})\n"
        f"coverage={rec['coverage_pct']:.1f}%   occ_rate={rec['occ_rate_pct']:.1f}%",
        fontsize=13, fontweight="bold", y=1.02
    )

    axes[0].imshow(orig)
    axes[0].set_title("Original satellite image", fontsize=11)
    axes[0].axis("off")

    axes[1].imshow(mask_to_rgb(pred_mask))
    axes[1].set_title("Predicted segmentation mask", fontsize=11)
    axes[1].axis("off")

    axes[2].imshow(orig)
    axes[2].imshow(mask_to_rgba(pred_mask))
    axes[2].set_title("Overlay  (green = empty · red = occupied)", fontsize=11)
    axes[2].axis("off")

    fig.legend(handles=legend_patches, loc="lower center", ncol=3, fontsize=11,
               bbox_to_anchor=(0.5, -0.04))
    plt.tight_layout()
    plt.show()

## 12 · Occupancy Rate Over Time

In [ ]:
header = f"{'Date':<13}  {'Background':>11}  {'Empty':>8}  {'Occupied':>9}  {'Occ. rate':>10}"
print(header)
print("─" * len(header))
for rec in results:
    print(
        f"{rec['label']:<13}  "
        f"{rec['bg_pct']:>10.1f}%  "
        f"{rec['empty_pct']:>7.1f}%  "
        f"{rec['occupied_pct']:>8.1f}%  "
        f"{rec['occ_rate_pct']:>9.1f}%"
    )

dated    = [r for r in results if r["date"] is not None]
dated    = sorted(dated, key=lambda r: r["date"])
dates    = [r["date"]        for r in dated]
occ_vals = [r["occ_rate_pct"] for r in dated]

if dated:
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(dates, occ_vals, marker="o", linewidth=2.5, markersize=9, color="firebrick")
    ax.fill_between(dates, occ_vals, alpha=0.12, color="firebrick")
    for d, v in zip(dates, occ_vals):
        ax.annotate(f"{v:.1f}%", (d, v), textcoords="offset points",
                    xytext=(0, 12), ha="center", fontsize=9)
    ax.set_xlabel("Date", fontsize=11)
    ax.set_ylabel("Occupancy rate", fontsize=11)
    ax.set_title("Walmart Parking Lot — Occupancy Rate Over Time (fine-tuned model)", fontsize=13)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0f}%"))
    ax.set_ylim(0, max(occ_vals) * 1.25 + 5)
    ax.grid(True, linestyle="--", alpha=0.45)
    plt.tight_layout()
    plt.savefig(CHECKPOINT_DIR / "walmart_ft_occupancy_trend.png", dpi=120)
    plt.show()

## 13 · Model Comparison: Exploratory vs Fine-Tuned

Load both checkpoints and run them side-by-side on the same satellite images
to see whether fine-tuning on walmart-1 improved predictions.

In [ ]:
# Load base model
base_model = smp.Unet(
    encoder_name    = "resnet34",
    encoder_weights = None,
    in_channels     = IN_CHANNELS,
    classes         = NUM_CLASSES,
    activation      = None,
)
base_model.load_state_dict(torch.load(PRETRAINED, map_location=DEVICE))
base_model = base_model.to(DEVICE)
base_model.eval()

def run_inference_with(m, img_path: Path):
    orig    = np.array(Image.open(img_path).convert("RGB"))
    h0, w0  = orig.shape[:2]
    resized = np.array(Image.fromarray(orig).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
    tensor  = preprocess(image=resized)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = m(tensor)
    pred_small = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
    return np.array(Image.fromarray(pred_small).resize((w0, h0), Image.NEAREST))

# Side-by-side comparison on each satellite image
for rec in results:
    orig      = rec["orig"]
    mask_ft   = rec["pred_mask"]
    mask_base = run_inference_with(base_model, rec["path"])

    pm_base = proxy_metrics(mask_base)
    pm_ft   = proxy_metrics(mask_ft)

    fig, axes = plt.subplots(1, 3, figsize=(21, 7))
    fig.suptitle(f"Model Comparison — {rec['label']}", fontsize=14, fontweight="bold", y=1.01)

    axes[0].imshow(orig)
    axes[0].set_title("Original image", fontsize=11)
    axes[0].axis("off")

    axes[1].imshow(orig)
    axes[1].imshow(mask_to_rgba(mask_base))
    axes[1].set_title(
        f"Exploratory (PKLot only)\n"
        f"cov {pm_base['coverage_pct']:.1f}%  occ {pm_base['occ_rate_pct']:.1f}%",
        fontsize=11
    )
    axes[1].axis("off")

    axes[2].imshow(orig)
    axes[2].imshow(mask_to_rgba(mask_ft))
    axes[2].set_title(
        f"Fine-tuned (+ walmart-1)\n"
        f"cov {pm_ft['coverage_pct']:.1f}%  occ {pm_ft['occ_rate_pct']:.1f}%",
        fontsize=11
    )
    axes[2].axis("off")

    fig.legend(handles=legend_patches, loc="lower center", ncol=3, fontsize=11,
               bbox_to_anchor=(0.5, -0.04))
    plt.tight_layout()
    plt.show()